# Cap-and-Trade Simulation — ECON7720 Lecture 04

## The setup

There are $n$ firms, each with **uncontrolled emissions** $E_0$ and a **linear marginal abatement cost**:

$$MAC_i(a_i) = s_i \cdot a_i$$

where $a_i = E_0 - e_i$ is firm $i$'s abatement and $s_i$ is its MAC slope. A steep slope means abatement is expensive; a flat slope means it's cheap.

## What the regulator does

1. **Sets a cap** $\bar{E}$ on total emissions (total abatement required: $A = n E_0 - \bar{E}$).
2. **Issues tradeable permits** — firms trade until the equimarginal condition holds:

$$MAC_1 = MAC_2 = \cdots = MAC_n = p$$

where $p$ is the **equilibrium permit price**. The cheap abater does more; the expensive abater buys permits instead.

## What the simulation shows

| Panel | What it shows |
|-------|---------------|
| **Left** | Each firm's MAC curve. Circles (●) = post-trade allocation (all at $p$). Crosses (×) = uniform standard (equal cuts, different MACs — inefficient). |
| **Centre** | Aggregate MAC (sum of individual demands to emit) meets the vertical cap → the permit price $p$ emerges. |
| **Right** | Total abatement cost: trading vs uniform standard. The gap is the **excess cost** of ignoring cost differences. |

## Parameters

| Slider | What it controls |
|--------|-----------------|
| **Cap** | Total allowed emissions $\bar{E}$. Lower cap → more abatement → higher permit price. |
| **Firms** | Number of sources $n$. More firms → more heterogeneity to exploit. |
| **Heterogeneity** | Spread of MAC slopes across firms. Zero = identical firms, no gains from trade. |
| **Base MAC slope** | Average steepness of abatement costs. Higher = abatement is generally more expensive. |
| **E₀ per firm** | Uncontrolled emissions per firm. Higher = more total emissions to regulate. |

In [ ]:
_HAS_WIDGETS = False
try:
    from ipywidgets import interact, IntSlider, FloatSlider
    _HAS_WIDGETS = True
    print("✓ Interactive sliders available.")
except ImportError:
    print("⚠ ipywidgets not available — use plot_simulation(cap=..., firms=..., heterogeneity=...) directly.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# ── UQ branding ──────────────────────────────────────────────────────
UQ_PURPLE = "#512478"
UQ_CYAN = "#0099CC"
COLORS = [UQ_PURPLE, UQ_CYAN, "#E6308A", "#2EA836", "#E87722", "#00A4BD",
          "#8B5CF6", "#DC2626"]


# ── Model ────────────────────────────────────────────────────────────

def make_slopes(n, base_slope, heterogeneity):
    """Generate n MAC slopes centred on base_slope with given spread."""
    if n == 1:
        return np.array([base_slope])
    spread = np.linspace(-heterogeneity, heterogeneity, n)
    return np.clip(base_slope + spread, 0.3, 15.0)


def solve_trading(slopes, cap, E0):
    """Cost-effective (equimarginal) allocation under trading."""
    n = len(slopes)
    total_abatement = max(n * E0 - cap, 0.0)
    inv_slopes = 1.0 / slopes
    permit_price = total_abatement / np.sum(inv_slopes)
    a_star = permit_price / slopes
    cost_trading = 0.5 * slopes * a_star**2
    return permit_price, a_star, cost_trading


def solve_uniform(slopes, cap, E0):
    """Uniform standard: each firm abates equally."""
    n = len(slopes)
    total_abatement = max(n * E0 - cap, 0.0)
    a_uniform = total_abatement / n
    cost_uniform = 0.5 * slopes * a_uniform**2
    return a_uniform, cost_uniform

In [ ]:
def plot_simulation(cap=24, firms=4, heterogeneity=1.5, base_slope=2.0, E0_per_firm=10):
    """Draw the three-panel cap-and-trade simulation."""
    n = int(firms)
    E0 = float(E0_per_firm)
    slopes = make_slopes(n, base_slope, heterogeneity)
    total_E0 = n * E0

    # Clamp cap to feasible range
    cap = min(cap, total_E0)

    permit_price, a_star, cost_trading = solve_trading(slopes, cap, E0)
    a_uniform, cost_uniform = solve_uniform(slopes, cap, E0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor("white")
    plt.subplots_adjust(wspace=0.32)
    fig.suptitle("Cap-and-Trade Simulation", fontsize=14,
                 fontweight="bold", color=UQ_PURPLE, y=1.02)

    # ── Panel 1: Individual firms ────────────────────────────────────
    ax1 = axes[0]
    ax1.set_title("Individual firms", fontsize=11, color=UQ_PURPLE)
    ax1.set_xlabel("Abatement $a_i$")
    ax1.set_ylabel("$ / unit")

    a_max = E0 * 1.1
    a_range = np.linspace(0, a_max, 200)

    for i in range(n):
        c = COLORS[i % len(COLORS)]
        ax1.plot(a_range, slopes[i] * a_range, color=c, lw=1.5, alpha=0.7,
                 label=f"Firm {i+1} ($s$={slopes[i]:.1f})")
        # Post-trade (circle)
        ax1.plot(a_star[i], permit_price, "o", color=c, ms=7, zorder=5)
        # Uniform (cross)
        ax1.plot(a_uniform, slopes[i] * a_uniform, "x", color=c,
                 ms=7, mew=2, zorder=5)

    ax1.axhline(permit_price, color=UQ_PURPLE, ls="--", lw=1, alpha=0.6,
                label=f"$p$ = {permit_price:.1f}")
    ax1.set_xlim(0, a_max)
    ax1.set_ylim(0, max(permit_price * 2.5, 5))
    ax1.legend(fontsize=7, loc="upper left", framealpha=0.8)

    # ── Panel 2: Permit market ───────────────────────────────────────
    ax2 = axes[1]
    ax2.set_title("Permit market", fontsize=11, color=UQ_PURPLE)
    ax2.set_xlabel("Emissions $E$")
    ax2.set_ylabel("$ / unit")

    inv_sum = np.sum(1.0 / slopes)
    E_range = np.linspace(0, total_E0, 300)
    agg_mac = np.maximum((total_E0 - E_range) / inv_sum, 0)

    ax2.plot(E_range, agg_mac, color=UQ_PURPLE, lw=2.5, label="Agg. MAC")
    ax2.axvline(cap, color=UQ_CYAN, lw=2.5, label=f"Cap = {cap:.0f}")

    # Shade abatement cost area under aggregate MAC from cap to total_E0
    E_fill = np.linspace(0, cap, 200)
    mac_fill = np.maximum((total_E0 - E_fill) / inv_sum, 0)
    ax2.fill_between(E_fill, mac_fill, alpha=0.08, color=UQ_PURPLE)

    ax2.plot([0, cap], [permit_price, permit_price], "--",
             color=UQ_PURPLE, lw=1, alpha=0.6)
    ax2.plot(cap, permit_price, "o", color=UQ_PURPLE, ms=8, zorder=5)
    ax2.annotate(f"$p$ = {permit_price:.1f}", xy=(cap, permit_price),
                 xytext=(cap + total_E0 * 0.08, permit_price + agg_mac[0]*0.08),
                 fontsize=9, color=UQ_PURPLE,
                 arrowprops=dict(arrowstyle="->", color=UQ_PURPLE, lw=1))

    # Mark total E0
    ax2.axvline(total_E0, color="gray", ls=":", lw=1, alpha=0.5)
    ax2.text(total_E0, agg_mac[0]*0.95, f"$E_0$={total_E0:.0f}",
             ha="right", fontsize=7, color="gray")

    ax2.set_xlim(0, total_E0 * 1.08)
    ax2.set_ylim(0, max(agg_mac[0] * 1.2, 5))
    ax2.legend(fontsize=8, loc="upper right", framealpha=0.8)

    # ── Panel 3: Cost comparison ─────────────────────────────────────
    ax3 = axes[2]
    ax3.set_title("Total abatement cost", fontsize=11, color=UQ_PURPLE)

    tc_trade = np.sum(cost_trading)
    tc_uniform = np.sum(cost_uniform)
    saving = tc_uniform - tc_trade
    saving_pct = 100 * saving / tc_uniform if tc_uniform > 0 else 0

    bars = ax3.bar(["Uniform\nstandard", "Cap-and-\ntrade"],
                   [tc_uniform, tc_trade],
                   color=[UQ_PURPLE, UQ_CYAN], width=0.5,
                   edgecolor="white", linewidth=1.5)

    # Excess-cost bracket
    if saving > 0.01:
        mid_y = (tc_uniform + tc_trade) / 2
        ax3.annotate("", xy=(0.28, tc_trade), xytext=(0.28, tc_uniform),
                     arrowprops=dict(arrowstyle="<->", color="#E6308A", lw=1.5))
        ax3.text(-0.15, mid_y, f"Excess cost\n\${saving:.0f} ({saving_pct:.0f}%)",
                 fontsize=8, color="#E6308A", ha="center", fontweight="bold",
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#E6308A",
                           alpha=0.9))

    ax3.set_ylabel("Total cost ($)")
    ax3.set_ylim(0, max(tc_uniform * 1.4, 1))

    for bar, val in zip(bars, [tc_uniform, tc_trade]):
        ax3.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + max(tc_uniform * 0.02, 0.3),
                 f"${val:.0f}", ha="center", va="bottom",
                 fontsize=9, fontweight="bold", color=UQ_PURPLE)

    plt.tight_layout()
    plt.show()

    # Print a summary table
    print(f"\n{'─'*60}")
    print(f"  Cap = {cap:.0f}  |  Firms = {n}  |  Total E₀ = {total_E0:.0f}")
    print(f"  Permit price p = {permit_price:.2f}")
    print(f"{'─'*60}")
    print(f"  {'Firm':<8} {'Slope':>6} {'Abate(trade)':>14} {'Abate(uniform)':>16} {'Emit(trade)':>13}")
    for i in range(n):
        print(f"  {i+1:<8} {slopes[i]:>6.2f} {a_star[i]:>14.2f} {a_uniform:>16.2f} {E0 - a_star[i]:>13.2f}")
    print(f"{'─'*60}")
    print(f"  Total cost (trading):  ${tc_trade:>10.1f}")
    print(f"  Total cost (uniform):  ${tc_uniform:>10.1f}")
    print(f"  Cost saving:           ${saving:>10.1f}  ({saving_pct:.1f}%)")
    print(f"{'─'*60}")

In [ ]:
if _HAS_WIDGETS:
    interact(
        plot_simulation,
        cap=IntSlider(value=24, min=2, max=100, step=1,
                      description="Cap (total E):",
                      style={"description_width": "initial"}),
        firms=IntSlider(value=4, min=2, max=8, step=1,
                        description="Number of firms:",
                        style={"description_width": "initial"}),
        heterogeneity=FloatSlider(value=1.5, min=0.0, max=4.0, step=0.1,
                                  description="MAC heterogeneity:",
                                  style={"description_width": "initial"}),
        base_slope=FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1,
                               description="Base MAC slope:",
                               style={"description_width": "initial"}),
        E0_per_firm=IntSlider(value=10, min=5, max=25, step=1,
                              description="E₀ per firm:",
                              style={"description_width": "initial"}),
    )
else:
    # No sliders — change the values below and re-run this cell to explore.
    plot_simulation(cap=24, firms=4, heterogeneity=1.5, base_slope=2.0, E0_per_firm=10)

## Things to try

### The basics
1. **Tighten the cap** (slide left): the permit price rises and costs increase — but trading *always* beats the uniform standard.
2. **Set heterogeneity to 0**: all firms are identical → trading saves nothing. This is the key insight: *no heterogeneity, no gains from trade*.
3. **Crank heterogeneity up**: the cost gap widens. More diverse firms = bigger gains from trade.

### Deeper exploration
4. **Set cap = total E₀** (n × E₀ per firm): no abatement needed → price = 0, costs = 0. The cap is not binding.
5. **Raise the base MAC slope**: abatement becomes more expensive for everyone. The permit price rises, but the *percentage* saving from trading stays roughly the same — it depends on heterogeneity, not on the level.
6. **Increase E₀ per firm**: more uncontrolled emissions to regulate. With the cap fixed, required abatement rises → higher price.
7. **Add more firms** (6–8): with more heterogeneous sources, the market has more room to exploit cost differences.

### Discussion questions
- Why does the cheap abater (flat MAC) do *more* abatement under trading but *less* under a uniform standard?
- If you were a regulator who didn't know the firms' MAC slopes, which instrument — uniform standard or cap-and-trade — would you prefer? Why?
- What happens to the permit price if a new low-cost abatement technology is invented (lower base slope)? Who gains?

---
*ECON7720 — Ecological & Environmental Economics | The University of Queensland | Dr Juan Soto-Diaz*